In [ ]:
import pandas as pd
import warnings
from model import NN
from utils import LoadData
from rolling_train_test import RollingTrainTest
import os
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
data_list = [
    'factor_structure_R.csv',
    'factor_cash_R.csv',
    'factor_debtpay_R.csv',
    'factor_divident_R.csv',
    'factor_growth_R.csv',
    'factor_profit_quality_R.csv',
    'factor_profit_R.csv',
    'factor_relative_R.csv',
    'factor_PS_R.csv',
    'factor_operatin_R.csv'
]

for _ in data_list:
    factor = pd.read_csv(f"../CSV/{_}")
    input_dim = factor.shape[1] - 2
    # print(f"Input dimension: {input_dim}")

    label = pd.read_csv('../CSV/label_cleaned.csv')
    # label.describe()
    target = 1

    # load data and model
    Data = LoadData(factor, label, batch_size=32, num_workers=0, shuffle=True)
    model_list = [
        NN(input_dim, target=target, alpha=0.8, l1_ratio=0.5, layer=3, model_name="NN"),
        NN(input_dim, target=target, alpha=0.8, l1_ratio=0.5, layer=4, model_name="NN"),
        NN(input_dim, target=target, alpha=0.8, l1_ratio=0.5, layer=5, model_name="NN")
    ]

    # rolling test
    count = 0
    return_list = []
    sr_list = []
    for model in model_list:
        RTT = RollingTrainTest(model, Data, train_size=0.5, test_size=0.1, epochs=20, patience=3, criterion=None, count=count)
        RTT.info(
            predictability_name = f"[--importance test--|--{_}--]"
            )
        RTT.run()
        RTT.backtest(trade_mode=2)
        count += 1
        print(f"Model {model.model_name} backtest completed...")
        return_list.append(RTT.Return)
        sr_list.append(RTT.SR)


    # 2\3\4\5层神经网络的平均值, mean_return
    # standard_Return = 1.2759
    standard_Return = round((1.2779+1.2764+1.2758+1.2735)/4, 4)
    standard_SR = round((4.4741+4.5023+4.4692+4.5027)/4, 4)

    test_Return = round(sum(return_list)/len(return_list), 4)
    test_SR = round(sum(sr_list)/len(sr_list), 4)
    importance_return = standard_Return - test_Return
    importance_sr = standard_SR - test_SR
    print("=" * 50)

    # write into CSV
    file_path = '../CSV_final/importance.csv'
    mode = 'a' if os.path.exists(file_path) else 'w'      
    with open(file_path, mode) as f:
        if mode == 'w':
            f.write('predictability_name,test_Return,importance_return,test_SR,importance_sr\n')
        f.write(f'{_},{test_Return:.4f},{importance_return:.4f},{test_SR:.4f},{importance_sr:.4f}\n')
